# Notebook 11: Approximate Nearest Neighbors

## Why This Matters
Exact nearest neighbor search is O(N·D) per query. For 1M documents at 384 dimensions, that's 384M multiplications per query — about 10ms on a fast CPU. At 100M documents it's a second. For real-time search this doesn't work.

ANN (Approximate Nearest Neighbor) algorithms build index structures that answer "what are the K nearest neighbors?" in O(log N) or O(1) with controllable recall–speed tradeoffs. FAISS is the production library — used in production at Meta, in Pinecone, in LangChain, and in most other vector search systems. Understanding it means understanding how all of them work.

## Structure
1. Why exact search doesn't scale (narrative)
2. FAISS flat index — exact brute force baseline
3. IVF — inverted file index, partition-and-search
4. HNSW — hierarchical navigable small worlds
5. Product quantization — compressing vectors
6. Recall vs speed tradeoff curves
7. Choosing an index for your use case
8. Bridge to vector databases

## What You'll Learn
- How IVF partitions the space to avoid searching all vectors
- How HNSW builds a multilayer graph for near-O(log N) search
- How product quantization compresses vectors by 4-64× with small recall loss
- How to plot recall vs QPS to choose the right index for your latency budget


## Background

### Why exact search doesn't scale

Exact search (computing distances to every vector) has:
- Time: O(N·D) per query
- Memory: O(N·D) floats
- No precomputation benefit beyond loading vectors

At 10M vectors × 768 dims × 4 bytes = 30GB just to store, 76B float multiplications per query.

### IVF — Inverted File Index

Partition the vector space into K clusters (Voronoi cells) using k-means. At query time:
1. Find the nearest cluster center(s) to the query — O(K)
2. Search only the vectors in those clusters — O(N/K per cluster × n_probe clusters)

Tradeoff: more clusters → faster search but higher recall loss if the true neighbor is in a different cell. `n_probe` controls how many cells to search at query time.

### HNSW — Hierarchical Navigable Small World

Build a multilayer graph where:
- Layer 0: all vectors, dense connections
- Layer L: fewer vectors, long-range connections (like highways)

Search: start from a random entry point at the top layer, greedily navigate toward the query, descend layers. Near O(log N) search time.

HNSW has excellent recall at high QPS and is the default in most production vector DBs. The downside: the graph is large in memory and slow to build.

### Product Quantization

Compress each D-dimensional vector by:
1. Split into M subvectors of D/M dimensions each
2. Quantize each subvector to one of 256 codewords (1 byte)
3. Store M bytes instead of D×4 bytes

Search using precomputed distance tables — fast, compact, small recall loss. Often combined with IVF: IVF+PQ is the standard high-scale configuration.


In [ ]:
# %pip install -q faiss-cpu sentence-transformers numpy matplotlib scikit-learn

In [ ]:
import numpy as np
import faiss
import time
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer

import torch
device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
print(f"FAISS version: {faiss.__version__}")
print(f"Device: {device}")

## 1. Building a Synthetic Corpus

In [ ]:
# Generate a corpus large enough to see ANN benefits
np.random.seed(42)
N_CORPUS = 10_000  # 10k docs — enough to compare index types
DIM = 384          # all-MiniLM-L6-v2 output dimension

# Simulate realistic embeddings: a mix of topic clusters
n_topics = 20
centers = np.random.randn(n_topics, DIM).astype(np.float32)
labels  = np.random.randint(0, n_topics, N_CORPUS)
noise   = np.random.randn(N_CORPUS, DIM).astype(np.float32) * 0.3
corpus_vecs = centers[labels] + noise

# Normalize to unit sphere (cosine similarity = dot product on unit vectors)
norms = np.linalg.norm(corpus_vecs, axis=1, keepdims=True)
corpus_vecs = corpus_vecs / norms

# Query vectors
N_QUERIES = 200
query_vecs = np.random.randn(N_QUERIES, DIM).astype(np.float32)
query_vecs /= np.linalg.norm(query_vecs, axis=1, keepdims=True)

print(f"Corpus: {N_CORPUS:,} vectors × {DIM} dims")
print(f"Queries: {N_QUERIES}")
print(f"Vector dtype: {corpus_vecs.dtype}")
print(f"Memory (corpus): {corpus_vecs.nbytes / 1e6:.1f} MB")

## 2. Flat Index — Exact Brute Force Baseline

In [ ]:
# IndexFlatIP: exact inner product (= cosine on unit vectors)
index_flat = faiss.IndexFlatIP(DIM)
index_flat.add(corpus_vecs)

t0 = time.perf_counter()
D_flat, I_flat = index_flat.search(query_vecs, k=10)
t_flat = (time.perf_counter() - t0) * 1000

print(f"Flat (exact) index:")
print(f"  Index size: {index_flat.ntotal:,} vectors")
print(f"  Search time ({N_QUERIES} queries): {t_flat:.1f} ms")
print(f"  Per-query: {t_flat/N_QUERIES:.2f} ms")
print(f"  Recall@10: 1.000 (exact by definition)")
print(f"  Memory: ~{corpus_vecs.nbytes/1e6:.1f} MB")

# Ground truth for recall computation
ground_truth = I_flat.copy()

def recall_at_k(approx_ids, exact_ids, k):
    """Fraction of exact top-k results that appear in approximate top-k."""
    total = 0
    for approx, exact in zip(approx_ids, exact_ids):
        total += len(set(approx[:k]) & set(exact[:k]))
    return total / (len(approx_ids) * k)

## 3. IVF — Inverted File Index

In [ ]:
results = {}  # collect for comparison plot

# IVF with L2: nlist clusters, n_probe cells searched per query
nlist = 100  # number of Voronoi cells
quantizer = faiss.IndexFlatIP(DIM)  # quantizer finds nearest cluster center
index_ivf = faiss.IndexIVFFlat(quantizer, DIM, nlist, faiss.METRIC_INNER_PRODUCT)

# IVF must be trained (k-means to find cluster centers)
index_ivf.train(corpus_vecs)
index_ivf.add(corpus_vecs)

print(f"IVF index: {nlist} clusters")
for n_probe in [1, 5, 10, 20, 50]:
    index_ivf.nprobe = n_probe
    t0 = time.perf_counter()
    D_ivf, I_ivf = index_ivf.search(query_vecs, k=10)
    t_ms = (time.perf_counter() - t0) * 1000
    rec = recall_at_k(I_ivf, ground_truth, 10)
    qps = N_QUERIES / (t_ms / 1000)
    results[f"IVF(nprobe={n_probe})"] = (rec, qps)
    print(f"  nprobe={n_probe:2d}: recall@10={rec:.3f}  {t_ms:.1f}ms  {qps:.0f} QPS")

## 4. HNSW — Hierarchical Navigable Small World

In [ ]:
# HNSW: M connections per node, efSearch controls search effort
M = 16  # connections per node in the graph
index_hnsw = faiss.IndexHNSWFlat(DIM, M, faiss.METRIC_INNER_PRODUCT)
index_hnsw.hnsw.efConstruction = 200  # higher = better graph quality, slower build

t_build = time.perf_counter()
index_hnsw.add(corpus_vecs)
build_time = time.perf_counter() - t_build

print(f"HNSW index: M={M}, built in {build_time:.1f}s")
print(f"  Graph levels: {index_hnsw.hnsw.max_level}")

for ef_search in [10, 20, 50, 100, 200]:
    index_hnsw.hnsw.efSearch = ef_search
    t0 = time.perf_counter()
    D_hnsw, I_hnsw = index_hnsw.search(query_vecs, k=10)
    t_ms = (time.perf_counter() - t0) * 1000
    rec = recall_at_k(I_hnsw, ground_truth, 10)
    qps = N_QUERIES / (t_ms / 1000)
    results[f"HNSW(ef={ef_search})"] = (rec, qps)
    print(f"  efSearch={ef_search:3d}: recall@10={rec:.3f}  {t_ms:.1f}ms  {qps:.0f} QPS")

## 5. Product Quantization

In [ ]:
# IVF + PQ: partition + quantize — best for large-scale, memory-constrained
M_pq = 8   # number of subvectors (DIM must be divisible by M_pq)
nbits = 8  # bits per subvector (256 codewords)

quantizer_pq = faiss.IndexFlatIP(DIM)
index_ivfpq  = faiss.IndexIVFPQ(quantizer_pq, DIM, nlist, M_pq, nbits)
index_ivfpq.train(corpus_vecs)
index_ivfpq.add(corpus_vecs)

# Memory savings
bytes_flat  = corpus_vecs.nbytes
bytes_pq    = N_CORPUS * M_pq  # M_pq bytes per vector
compression = bytes_flat / bytes_pq

print(f"IVF+PQ index (M={M_pq}, nbits={nbits}):")
print(f"  Flat memory:   {bytes_flat/1e6:.1f} MB")
print(f"  PQ memory:     {bytes_pq/1e6:.2f} MB")
print(f"  Compression:   {compression:.0f}×")

for n_probe in [5, 10, 20]:
    index_ivfpq.nprobe = n_probe
    t0 = time.perf_counter()
    D_pq, I_pq = index_ivfpq.search(query_vecs, k=10)
    t_ms = (time.perf_counter() - t0) * 1000
    rec = recall_at_k(I_pq, ground_truth, 10)
    qps = N_QUERIES / (t_ms / 1000)
    results[f"IVF+PQ(nprobe={n_probe})"] = (rec, qps)
    print(f"  nprobe={n_probe:2d}: recall@10={rec:.3f}  {t_ms:.1f}ms  {qps:.0f} QPS")

## 6. Recall vs QPS Tradeoff Curve

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

# Group by index type
groups = {
    "IVF":    {k:v for k,v in results.items() if k.startswith("IVF(")},
    "HNSW":   {k:v for k,v in results.items() if k.startswith("HNSW")},
    "IVF+PQ": {k:v for k,v in results.items() if k.startswith("IVF+PQ")},
}
colors_g = {"IVF":"royalblue","HNSW":"seagreen","IVF+PQ":"darkorange"}

for gname, gresults in groups.items():
    recs = [v[0] for v in gresults.values()]
    qpss = [v[1] for v in gresults.values()]
    ax.plot(qpss, recs, 'o-', color=colors_g[gname], label=gname, linewidth=2)

# Flat baseline
flat_qps = N_QUERIES / (t_flat / 1000)
ax.scatter([flat_qps], [1.0], marker='*', s=200, color='red', zorder=5, label=f"Flat (exact)")

ax.set_xlabel("Queries per second (QPS) — higher is faster")
ax.set_ylabel("Recall@10 — higher is better")
ax.set_title("Recall vs Speed Tradeoff by Index Type")
ax.legend(); ax.grid(True, alpha=0.3)
ax.set_xlim(left=0)
ax.set_ylim([0.5, 1.05])
plt.tight_layout(); plt.show()

## 7. Choosing an Index

| Scale | Constraint | Recommended |
|-------|-----------|-------------|
| < 100k vectors | Any | IndexFlatIP (exact, simple) |
| 100k – 1M | Memory OK | HNSW (best recall/speed, no train step) |
| 100k – 1M | Memory tight | IVF+PQ (compression, lower recall) |
| > 1M | Memory OK | IVF + HNSW quantizer |
| > 1M | Memory tight | IVF+PQ or ScaNN |

**Rule of thumb for nlist (IVF):** `nlist = 4 × sqrt(N)` is a common starting point. More clusters = faster search, but training requires more vectors and recall drops faster with low n_probe.

**HNSW parameters:**
- `M`: 8–64, higher = better recall, more memory, slower build
- `efConstruction`: 100–500, higher = better graph, slower build
- `efSearch`: tune at query time for your recall budget


## 8. Bridge to Vector Databases

FAISS is a library — you manage the index yourself: add vectors, persist to disk, handle updates. Vector databases wrap this (and other ANN implementations) with:
- REST APIs for add/search/delete
- Metadata filtering alongside vector search
- Distributed scaling across machines
- Automatic persistence and replication
- Built-in embedding model integration

Notebook 12 covers Chroma (lightweight, good for development) and the architectural tradeoffs between pgvector, Chroma, Pinecone, and Weaviate.


## Summary

| Index | Build | Memory | Recall | Speed | Best for |
|-------|-------|--------|--------|-------|----------|
| Flat | O(1) | O(N·D) | 1.0 | O(N) | < 100k, baseline |
| IVF | Train | O(N·D) | 0.9–0.99 | O(N/nlist) | 100k–10M, tunable |
| HNSW | O(N log N) | O(N·M·D) | 0.95–0.999 | O(log N) | Default for most |
| IVF+PQ | Train | O(N·M_pq) | 0.8–0.95 | O(N/nlist) | Large scale, memory-constrained |

**Next:** Notebook 12 — Vector Databases. Chroma for development, pgvector for production Postgres users, and when to move to a dedicated vector DB.
